# 객체 탐지(YOLO)만 학습 (Colab)

주행 CNN 없이 노루·멧돼지 탐지 모델만 학습합니다. 주행과 함께 학습하려면 `train_all.ipynb` 를 씁니다.

1. Pi 에서 라벨링하고 묶기: `cd aicar && python3 label.py` → `python3 label.py --zip` → `yolo_dataset.zip`
   (Roboflow 에서 YOLOv8 형식으로 내보낸 zip 도 됩니다)
2. Colab 파일 창에 zip 하나를 올립니다.
3. 런타임 → 런타임 유형 변경 → T4 GPU → 저장, 그다음 런타임 → 모두 실행
4. 끝나면 `detect_models.zip` 이 내려받아집니다. Pi 의 `aicar/` 에 옮기고 주행에 쓰는 프로젝트 폴더에 풉니다.
   ```
   cd aicar && unzip -o detect_models.zip -d wayA      # wayA/ 안에 animal_yolo26n_ncnn/, person_yolo26n_ncnn/
   sudo python3 ar.py wayA                             # 시작할 때 "[탐지] YOLO 모델을 읽었습니다"
   ```
5. 4번 셀의 test 재현율(recall)을 `yolo/models/README.md` 표에 적습니다 (목표 85% 이상).

세션이 끊겨도 결과가 남도록 Drive(`MyDrive/aicar_models/`)에 복사합니다.

## 1. Drive 연결 (결과 백업용)

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
BACKUP_DIR = '/content/drive/MyDrive/aicar_models'
os.makedirs(BACKUP_DIR, exist_ok=True)

def backup(path):
    '''학습 결과를 Drive 로 복사합니다 (파일 또는 폴더).'''
    dst = os.path.join(BACKUP_DIR, os.path.basename(path))
    if os.path.isdir(path):
        shutil.copytree(path, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(path, dst)
    print('Drive 백업:', dst)


## 2. 준비

In [ ]:
# 2026-11-12 모델과 같은 버전
!pip install -q ultralytics==8.4.211

In [ ]:
import zipfile, yaml
from ultralytics import YOLO

YOLO_EPOCHS = 100      # 사진이 적으면 50~100, 많으면 늘림
ANIMAL_IMGSZ = 640     # aicar/yolo_detect.py ANIMAL_IMGSZ 와 같아야 함
PERSON_IMGSZ = 416     # aicar/yolo_detect.py PERSON_IMGSZ 와 같아야 함
YOLO_DIR = '/content/yolo_dataset'
NAMES = {0: 'person', 1: 'roe_deer', 2: 'wild_boar'}   # aicar/label.py, yolo/train/data.yaml 과 같은 순서

zips = [f for f in os.listdir('.') if f.endswith('.zip') and not f.endswith('_models.zip')]
assert len(zips) == 1, f'데이터셋 zip 을 하나만 올려 주세요 (label.py --zip 또는 Roboflow YOLOv8): {zips}'
print('데이터셋:', zips[0])
shutil.rmtree(YOLO_DIR, ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(YOLO_DIR)

# label.py: images/train, images/val, images/test   /   Roboflow: train/images, valid/images, test/images
if os.path.isdir(os.path.join(YOLO_DIR, 'images', 'train')):
    split = {'train': 'images/train', 'val': 'images/val', 'test': 'images/test'}
else:
    split = {'train': 'train/images', 'val': 'valid/images', 'test': 'test/images'}
for k, rel in split.items():
    d = os.path.join(YOLO_DIR, rel)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'{k}: {n}장')
    assert n > 0, f'{rel} 가 비어 있습니다'

# Roboflow zip 이면 클래스 순서가 같은지 확인 (다르면 번호가 섞여 노루를 사람으로 배움)
rf = os.path.join(YOLO_DIR, 'data.yaml')
if os.path.exists(rf):
    names = yaml.safe_load(open(rf))['names']
    names = list(names.values()) if isinstance(names, dict) else list(names)
    assert names == list(NAMES.values()), \
        f'클래스 순서가 다릅니다: {names}. Roboflow 에 person, roe_deer, wild_boar 세 클래스를 모두 만들어 주세요'

with open('animal_data.yaml', 'w') as f:
    yaml.safe_dump({'path': YOLO_DIR, **split, 'names': NAMES}, f, allow_unicode=True)

## 3. 학습

In [ ]:
animal = YOLO('yolo26n.pt')
animal.train(data='animal_data.yaml', imgsz=ANIMAL_IMGSZ, epochs=YOLO_EPOCHS, project='/content/runs', name='animal', exist_ok=True)
best = YOLO('/content/runs/animal/weights/best.pt')
backup('/content/runs/animal/weights/best.pt')


## 4. 평가 (학습에 쓰지 않은 test 세트)

In [ ]:
# 학습에 쓰지 않은 test 세트로 평가 → 클래스별 재현율을 yolo/models/README.md 에 기록 (목표 85% 이상)
metrics = best.val(data='animal_data.yaml', imgsz=ANIMAL_IMGSZ, split='test')
for i, name in best.names.items():
    if i in metrics.box.ap_class_index:
        k = list(metrics.box.ap_class_index).index(i)
        print(f'{name:10s} recall={metrics.box.r[k]:.3f}  mAP50={metrics.box.ap50[k]:.3f}')


## 5. Raspberry Pi 용 변환과 내려받기

In [ ]:
# Raspberry Pi 용 NCNN 변환
animal_ncnn = best.export(format='ncnn', imgsz=ANIMAL_IMGSZ)
person_ncnn = YOLO('yolo26n.pt').export(format='ncnn', imgsz=PERSON_IMGSZ)
print(animal_ncnn, person_ncnn)


In [ ]:
NCNN_FILES = ('model.ncnn.param', 'model.ncnn.bin', 'metadata.yaml')
out = 'detect_models.zip'
with zipfile.ZipFile(out, 'w') as z:
    for src_dir, name in ((animal_ncnn, 'animal_yolo26n_ncnn'), (person_ncnn, 'person_yolo26n_ncnn')):
        for f in NCNN_FILES:
            z.write(os.path.join(src_dir, f), f'{name}/{f}')   # aicar/yolo_detect.py 가 읽는 폴더 이름
for n in zipfile.ZipFile(out).namelist():
    print(n)
backup(out)
from google.colab import files
files.download(out)